In [0]:
%sql
create table if not exists workspace.bookstore_eng_pro.orders_silver
(order_id STRING, order_timestamp timestamp, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>)

In [0]:
from pyspark.sql import functions as F

orders_schema = "order_id STRING, order_timestamp timestamp, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"
deduped_df = (spark.readStream.table("workspace.bookstore_eng_pro.bronze")
                                .where("topic = 'orders'")
                                .select(F.from_json(F.unbase64(F.col("value")).cast("string"),orders_schema).alias("v"))
                                .select("v.*")
                                .withWatermark("order_timestamp", "30 seconds")
                                .dropDuplicates(["order_id", "order_timestamp"]))

In [0]:
def upsert_data(microBatchDF, batch):
    microBatchDF.createOrReplaceTempView("orders_microbatch")

    sql_query = """
              MERGE INTO workspace.bookstore_eng_pro.orders_silver a
              USING orders_microbatch b
              on a.order_id = b.order_id and a.order_timestamp = b.order_timestamp
              WHEN NOT MATCHED THEN INSERT *
              """
    microBatchDF.sparkSession.sql(sql_query)

In [0]:
query =  (deduped_df.writeStream
                     .foreachBatch(upsert_data)
                     .option("checkpointLocation", "/Volumes/workspace/bookstore_eng_pro/checkpoints/orders_silver")
                     .trigger(availableNow=True)
                     .start())
query.awaitTermination()

In [0]:
streaming_total = spark.read.table("workspace.bookstore_eng_pro.orders_silver").count()

print(f"Streaming total: {streaming_total}")